In [83]:
%run "ms_lms_config"

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 90, Finished, Available, Finished)

In [84]:
today_date = ''

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 91, Finished, Available, Finished)

### Dimensions

#### Dim Student

In [85]:
df = spark.read.table("ms_lms_lh_200_silver.lms_200_silver")

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 92, Finished, Available, Finished)

In [86]:
df.printSchema()

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 93, Finished, Available, Finished)

root
 |-- Student_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = false)
 |-- Gender: string (nullable = false)
 |-- Grade_Level: string (nullable = true)
 |-- Course_ID: string (nullable = true)
 |-- Course_Name: string (nullable = true)
 |-- Enrollment_Date: date (nullable = true)
 |-- Completion_Date: date (nullable = true)
 |-- Status: string (nullable = false)
 |-- Final_Grade: string (nullable = false)
 |-- Attendance_Rate: double (nullable = false)
 |-- Time_Spent_on_Course_hrs: double (nullable = false)
 |-- Assignments_Completed: integer (nullable = false)
 |-- Quizzes_Completed: integer (nullable = false)
 |-- Forum_Posts: integer (nullable = false)
 |-- Messages_Sent: integer (nullable = false)
 |-- Quiz_Average_Score: double (nullable = false)
 |-- Assignment_Scores: string (nullable = true)
 |-- Assignment_Average_Score: double (nullable = false)
 |-- Project_Score: double (nullable = false)
 |-- Extra_Credit: double (nullable

In [87]:
student_schema = StructType([
    StructField("Student_ID", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Demographic_Group", StringType(), True),
    StructField("Internet_Access", StringType(), True),
    StructField("Learning_Disabilities", StringType(), True),
    StructField("Preferred_Learning_Style", StringType(), True),
    StructField("Language_Proficiency", StringType(), True),
    StructField("Parent_Involvement", StringType(), True)
])

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 94, Finished, Available, Finished)

In [88]:
DeltaTable \
.createIfNotExists(spark) \
.tableName('student') \
.addColumns(student_schema) \
.execute()

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 95, Finished, Available, Finished)

In [89]:
# if not spark.catalog.tableExists("Student"):
#     df = spark.createDataFrame([], student_schema)  # pusty DataFrame z definicją schematu
#     df.write.format("delta").mode("overwrite").saveAsTable("student")
# else:
#     print("Table already exists.")


StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 96, Finished, Available, Finished)

#### Dim Course

In [90]:
course_schema = StructType([
    StructField("Course_ID", StringType(), True),
    StructField("Course_Name", StringType(), True),
    StructField("Grade_Level", StringType(), True)
])

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 97, Finished, Available, Finished)

In [91]:
DeltaTable \
.createIfNotExists(spark) \
.tableName('course') \
.addColumns(course_schema) \
.execute()

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 98, Finished, Available, Finished)

### Fact Student Performance

In [92]:
student_performance_schema = StructType([
    StructField("Student_ID", StringType(), True),
    StructField("Course_ID", StringType(), True),
    StructField("Enrollment_Date", DateType(), True),
    StructField("Completion_Date", DateType(), True),
    StructField("Status", StringType(), False),
    StructField("Final_Grade", StringType(), False),
    StructField("Attendance_Rate", DoubleType(), False),
    StructField("Time_Spent_on_Course_hrs", DoubleType(), False),
    StructField("Assignments_Completed", IntegerType(), False),
    StructField("Quizzes_Completed", IntegerType(), False),
    StructField("Forum_Posts", IntegerType(), False),
    StructField("Messages_Sent", IntegerType(), False),
    StructField("Quiz_Average_Score", DoubleType(), False),
    StructField("Assignment_Scores", StringType(), True),
    StructField("Assignment_Average_Score", DoubleType(), False),
    StructField("Project_Score", DoubleType(), False),
    StructField("Extra_Credit", DoubleType(), False),
    StructField("Overall_Performance", DoubleType(), False),
    StructField("Feedback_Score", DoubleType(), False),
    StructField("Completion_Time_Days", IntegerType(), True),
    StructField("Performance_Score", DoubleType(), False),
    StructField("Course_Completion_Rate", StringType(), False),
    StructField("Processing_Date", DateType(), True)
])


StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 99, Finished, Available, Finished)

In [93]:

DeltaTable.createIfNotExists(spark).tableName("student_performance")\
           .addColumns(student_performance_schema)\
           .execute()

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 100, Finished, Available, Finished)

#### Load Dim Student

In [94]:
df_200_silver = spark.read.table('ms_lms_lh_200_silver.lms_200_silver')

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 101, Finished, Available, Finished)

In [95]:
df_200_silver_student = df_200_silver \
    .select(
        "Student_ID", 
        "Name", 
        "Age", 
        "Gender", 
        "Demographic_Group", 
        "Internet_Access", 
        "Learning_Disabilities", 
        "Preferred_Learning_Style", 
        "Language_Proficiency", 
        "Parent_Involvement"
    ) \
    .dropDuplicates()


StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 102, Finished, Available, Finished)

In [96]:
delta_300_gold_student_path = 'abfss://30b7edc1-64f9-4583-8c8f-2498c1953029@onelake.dfs.fabric.microsoft.com/bdde4fe9-3159-4d50-930c-709465eb7838/Tables/student'
delta_300_gold_student = DeltaTable.forPath(spark,delta_300_gold_student_path)

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 103, Finished, Available, Finished)

In [97]:
delta_300_gold_student \
.alias('target') \
.merge(
   df_200_silver_student.alias('source'),
   'target.Student_ID = source.Student_ID' 
) \
.whenMatchedUpdate(
    set = {
    "Name": "source.Name",
    "Age": "source.Age",
    "Gender": "source.Gender",
    "Demographic_Group": "source.Demographic_Group",
    "Internet_Access": "source.Internet_Access",
    "Learning_Disabilities": "source.Learning_Disabilities",
    "Preferred_Learning_Style": "source.Preferred_Learning_Style",
    "Language_Proficiency": "source.Language_Proficiency",
    "Parent_Involvement": "source.Parent_Involvement"
    }
) \
.whenNotMatchedInsert(
    values = {
    "Student_ID": "source.Student_ID",
    "Name": "source.Name",
    "Age": "source.Age",
    "Gender": "source.Gender",
    "Demographic_Group": "source.Demographic_Group",
    "Internet_Access": "source.Internet_Access",
    "Learning_Disabilities": "source.Learning_Disabilities",
    "Preferred_Learning_Style": "source.Preferred_Learning_Style",
    "Language_Proficiency": "source.Language_Proficiency",
    "Parent_Involvement": "source.Parent_Involvement"
    }
) \
.execute()

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 104, Finished, Available, Finished)

In [98]:
# history_student = delta_300_gold_student \
# .history(1) \
# .select('operationMetrics') \
# .collect()[0][0]

# display(history_student)

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 105, Finished, Available, Finished)

#### Load Dim Course

In [99]:
df_200_silver_course = df_200_silver \
.select(
    "Course_ID", 
    "Course_Name", 
    "Grade_Level"
) \
.dropDuplicates()

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 106, Finished, Available, Finished)

In [100]:
delta_300_gold_course_path = 'abfss://30b7edc1-64f9-4583-8c8f-2498c1953029@onelake.dfs.fabric.microsoft.com/bdde4fe9-3159-4d50-930c-709465eb7838/Tables/course'
delta_300_gold_course = DeltaTable.forPath(spark,delta_300_gold_course_path)

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 107, Finished, Available, Finished)

In [101]:
delta_300_gold_course \
.alias('target') \
.merge(
    df_200_silver_course.alias('source'),
   'target.Course_ID = source.Course_ID'
) \
.whenMatchedUpdate(
    set={
    "Course_ID": "source.Course_ID",
    "Course_Name": "source.Course_Name",
    "Grade_Level": "source.Grade_Level"
    }
) \
.whenNotMatchedInsert(
    values={
    "Course_ID": "source.Course_ID",
    "Course_Name": "source.Course_Name",
    "Grade_Level": "source.Grade_Level"
    }
) \
.execute()



StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 108, Finished, Available, Finished)

#### Load Fact Student Performance

In [102]:
df_200_silver_student_performance = df_200_silver \
.select(
    "Student_ID",
    "Course_ID",
    "Enrollment_Date",
    "Completion_Date",
    "Status",
    "Final_Grade",
    "Attendance_Rate",
    "Time_Spent_on_Course_hrs",
    "Assignments_Completed",
    "Quizzes_Completed",
    "Forum_Posts",
    "Messages_Sent",
    "Quiz_Average_Score",
    "Assignment_Scores",
    "Assignment_Average_Score",
    "Project_Score",
    "Extra_Credit",
    "Overall_Performance",
    "Feedback_Score",
    "Completion_Time_Days",
    "Performance_Score",
    "Course_Completion_Rate",
    "Processing_Date"
)

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 109, Finished, Available, Finished)

In [103]:
delta_300_gold_student_performance_path = 'abfss://30b7edc1-64f9-4583-8c8f-2498c1953029@onelake.dfs.fabric.microsoft.com/bdde4fe9-3159-4d50-930c-709465eb7838/Tables/student_performance'
delta_300_gold_student_performance = DeltaTable.forPath(spark,delta_300_gold_student_performance_path)

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 110, Finished, Available, Finished)

In [104]:
delta_300_gold_student_performance \
.alias('target') \
.merge(
   df_200_silver_student_performance.alias('source'),
   'target.Student_ID = source.Student_ID AND target.Course_ID = source.Course_ID' 
) \
.whenMatchedUpdate(
    set = {
    "Student_ID": "source.Student_ID",
    "Course_ID": "source.Course_ID",
    "Enrollment_Date": "source.Enrollment_Date",
    "Completion_Date": "source.Completion_Date",
    "Status": "source.Status",
    "Final_Grade": "source.Final_Grade",
    "Attendance_Rate": "source.Attendance_Rate",
    "Time_Spent_on_Course_hrs": "source.Time_Spent_on_Course_hrs",
    "Assignments_Completed": "source.Assignments_Completed",
    "Quizzes_Completed": "source.Quizzes_Completed",
    "Forum_Posts": "source.Forum_Posts",
    "Messages_Sent": "source.Messages_Sent",
    "Quiz_Average_Score": "source.Quiz_Average_Score",
    "Assignment_Scores": "source.Assignment_Scores",
    "Assignment_Average_Score": "source.Assignment_Average_Score",
    "Project_Score": "source.Project_Score",
    "Extra_Credit": "source.Extra_Credit",
    "Overall_Performance": "source.Overall_Performance",
    "Feedback_Score": "source.Feedback_Score",
    "Completion_Time_Days": "source.Completion_Time_Days",
    "Performance_Score": "source.Performance_Score",
    "Course_Completion_Rate": "source.Course_Completion_Rate",
    "Processing_Date": "source.Processing_Date"
    }
) \
.whenNotMatchedInsert(
    values = {
    "Student_ID": "source.Student_ID",
    "Course_ID": "source.Course_ID",
    "Enrollment_Date": "source.Enrollment_Date",
    "Completion_Date": "source.Completion_Date",
    "Status": "source.Status",
    "Final_Grade": "source.Final_Grade",
    "Attendance_Rate": "source.Attendance_Rate",
    "Time_Spent_on_Course_hrs": "source.Time_Spent_on_Course_hrs",
    "Assignments_Completed": "source.Assignments_Completed",
    "Quizzes_Completed": "source.Quizzes_Completed",
    "Forum_Posts": "source.Forum_Posts",
    "Messages_Sent": "source.Messages_Sent",
    "Quiz_Average_Score": "source.Quiz_Average_Score",
    "Assignment_Scores": "source.Assignment_Scores",
    "Assignment_Average_Score": "source.Assignment_Average_Score",
    "Project_Score": "source.Project_Score",
    "Extra_Credit": "source.Extra_Credit",
    "Overall_Performance": "source.Overall_Performance",
    "Feedback_Score": "source.Feedback_Score",
    "Completion_Time_Days": "source.Completion_Time_Days",
    "Performance_Score": "source.Performance_Score",
    "Course_Completion_Rate": "source.Course_Completion_Rate",
    "Processing_Date": "source.Processing_Date"
    }
) \
.execute()

StatementMeta(, 869f168a-0c88-46d0-9962-abac862bbdcb, 111, Finished, Available, Finished)